In [16]:
from pathlib import Path
pdf_paths = sorted(Path("../data/raw").glob("*.pdf"))
print(pdf_paths)

[WindowsPath('../data/raw/potato_disease-pest-management_cpri_2016_manual.pdf'), WindowsPath('../data/raw/rice_weed-mgmt-dry-direct-seeded_nrri_2019_brief.pdf'), WindowsPath('../data/raw/rice_weedy-rice-options_dwsr_2013_article.pdf'), WindowsPath('../data/raw/wheat_rust-resistant-varieties-economics_vpkas_2020_article.pdf')]


In [ ]:

# from langchain_community.document_loaders import PyPDFLoader

# def meta_from_name(path):
#     crop, topic, org, year, kind = path.stem.split("_")
#     return {"doc_id": path.stem, "crop": crop, "topic": topic,
#             "org": org, "year": int(year), "kind": kind}

# pages = []
# for p in pdf_paths:
#     docs = PyPDFLoader(str(p)).load()
#     for d in docs:
#         d.metadata.update(meta_from_name(p))
#     print(p.name, len(docs), "pages")
#     pages.extend(docs)

C:\Users\aryan\AppData\Local\Temp\ipykernel_6024\53856542.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
Advanced encoding /SymbolSetEncoding not implemented yet, using ['\x00', '\x01', '\x02', '\x03', '\x04', '\x05', '\x06', '\x07', '\x08', '\t', '\n', '\x0b', '\x0c', '\r', '\x0e', '\x0f', '\x10', '\x11', '\x12', '\x13', '\x14', '\x15', '\x16', '\x17', '\x18', '\x19', '\x1a', '\x1b', '\x1c', '\x1d', '\x1e', '\x1f', ' ', '!', '"', '#', '$', '%', '&', '’', '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_',

potato_disease-pest-management_cpri_2016_manual.pdf.pdf 142 pages
rice_weed-mgmt-dry-direct-seeded_nrri_2019_brief.pdf.pdf 8 pages
rice_weedy-rice-options_dwsr_2013_article.pdf.pdf 6 pages
wheat_rust-resistant-varieties-economics_vpkas_2020_article.pdf.pdf 8 pages


In [17]:
import pymupdf
from langchain_core.documents import Document

KINDS = {"brief", "bulletin", "manual", "article", "guideline", "leaflet"}

def meta_from_name(path):
    crop, topic, org, year, kind = path.stem.split("_")
    assert kind in KINDS, f"unexpected kind {kind!r} in {path.name}"
    return {"doc_id": path.stem, "crop": crop, "topic": topic,
            "org": org, "year": int(year), "kind": kind}

def load_pdf(path):
    docs = []
    with pymupdf.open(str(path)) as pdf:
        for i, page in enumerate(pdf):
            meta = meta_from_name(path)
            meta.update({"source": path.name, "page": i + 1})
            docs.append(Document(page_content=page.get_text(), metadata=meta))
    return docs

pages = []
for p in pdf_paths:
    docs = load_pdf(p)
    print(p.name, len(docs), "pages")
    pages.extend(docs)

potato_disease-pest-management_cpri_2016_manual.pdf 142 pages
rice_weed-mgmt-dry-direct-seeded_nrri_2019_brief.pdf 8 pages
rice_weedy-rice-options_dwsr_2013_article.pdf 6 pages
wheat_rust-resistant-varieties-economics_vpkas_2020_article.pdf 8 pages


In [18]:
print(pages[5].metadata)
print(pages[5].page_content[:1000])

empty = [d for d in pages if len(d.page_content.strip()) < 50]
print(len(empty), "of", len(pages), "pages have almost no text")

{'doc_id': 'potato_disease-pest-management_cpri_2016_manual', 'crop': 'potato', 'topic': 'disease-pest-management', 'org': 'cpri', 'year': 2016, 'kind': 'manual', 'source': 'potato_disease-pest-management_cpri_2016_manual.pdf', 'page': 6}
CONTENTS
Chapter
No.
Title
Contributors
Pages
13.
Nematodes and their management
ARTI BAIRWA, VENKATASALM E P,
PRIYANK H M & SUDHA
ICAR-C.P.R.I, Shimla – 171 001
68 – 74
14.
Pesticide resistance management
GAURAV VERMA
Scientist
Division of Plant Protection
ICAR-C.P.R.I, Shimla – 171 001
75 – 88
15.
Disease forecasting for early and late blight of
potato
SANJEEV SHARMA
Senior Scientist
Division of Plant Protection
ICAR-C.P.R.I, Shimla – 171 001
89 – 92
16.
Forecasting of insect vectors for management
of potato virus diseases
SRIDHAR J, V VENKATESWARLU,
KAMLESH MALIK, ANUJ
BHATNAGAR & M. ABAS SHAH
Division of Plant Protection
ICAR-C.P.R.I, Shimla – 171 001
93 – 98
17.
Role of nutrients in potato health
JAGDEV SHARMA
Principal Scientist
Division of Crop

In [19]:
from collections import Counter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(pages)

actual = Counter((c.metadata["doc_id"], c.metadata["page"]) for c in chunks)
diffs = []
for d in pages:
    key = (d.metadata["doc_id"], d.metadata["page"])
    L = len(d.page_content)
    pred = 1 if L <= 1000 else 1 + math.ceil((L - 1000) / 900)
    diffs.append((actual[key] - pred, key, L))

print(Counter(x[0] for x in diffs))
print(sorted(diffs)[:5])

Counter({0: 160, -1: 4})
[(-1, ('potato_disease-pest-management_cpri_2016_manual', 17), 4643), (-1, ('potato_disease-pest-management_cpri_2016_manual', 60), 4621), (-1, ('potato_disease-pest-management_cpri_2016_manual', 95), 3720), (-1, ('potato_disease-pest-management_cpri_2016_manual', 100), 4697), (0, ('potato_disease-pest-management_cpri_2016_manual', 1), 169)]


In [8]:
import math
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_stats(pages, size, overlap):
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    chunks = splitter.split_documents(pages)
    lens = [len(c.page_content) for c in chunks]
    print(size, overlap, "chunks:", len(chunks),
          "min/avg/max:", min(lens), sum(lens)//len(lens), max(lens))
    return chunks

def predicted(pages, size, overlap):
    total = 0
    for d in pages:
        L = len(d.page_content)
        if L == 0:
            continue
        total += 1 if L <= size else 1 + math.ceil((L - size) / (size - overlap))
    return total

for size, overlap in [(500, 50), (1000, 100), (1500, 150), (1000, 0)]:
    chunk_stats(pages, size, overlap)
    print("   predicted by formula:", predicted(pages, size, overlap))

500 50 chunks: 1252 min/avg/max: 25 419 500
   predicted by formula: 1224
1000 100 chunks: 641 min/avg/max: 63 835 1000
   predicted by formula: 645
1500 150 chunks: 461 min/avg/max: 163 1200 1500
   predicted by formula: 460
1000 0 chunks: 629 min/avg/max: 60 827 1000
   predicted by formula: 614


In [9]:
chunks = chunk_stats(pages, 1000, 100)
import random
for c in random.sample(chunks, 3):
    print(c.metadata)
    print(c.page_content)
    print("-" * 60)

1000 100 chunks: 641 min/avg/max: 63 835 1000
{'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2020-10-18T08:45:11+05:30', 'source': '..\\data\\raw\\wheat_rust-resistant-varieties-economics_vpkas_2020_article.pdf.pdf', 'file_path': '..\\data\\raw\\wheat_rust-resistant-varieties-economics_vpkas_2020_article.pdf.pdf', 'total_pages': 8, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2020-10-18T08:45:12+05:30', 'trapped': '', 'modDate': "D:20201018084512+05'30'", 'creationDate': "D:20201018084511+05'30'", 'page': 7, 'doc_id': 'wheat_rust-resistant-varieties-economics_vpkas_2020_article.pdf', 'crop': 'wheat', 'topic': 'rust-resistant-varieties-economics', 'org': 'vpkas', 'year': 2020, 'kind': 'article.pdf'}
pigs in the Indian Himalayan region. Current Science 
116(6): 1015-1019.
19.	 Samui SK, S Mitra, DK Roy, AK Mandal and D 
Saha. 2000. Evaluation of frontline demonstration 
on groundnut. Journ